# 🎯 Real-Time Face Recognition with YOLO + face_recognition

## Overview
This notebook builds a **real-time face recognition system** that combines two powerful tools:

| Component | Role |
|---|---|
| **YOLOv8** | Fast object/face detection in each webcam frame |
| **face_recognition** | Deep-learning face encoding & identity matching |
| **OpenCV** | Webcam capture and drawing bounding boxes |
| **PIL (Pillow)** | Image saving & format conversion |

### How the Pipeline Works
```
Webcam Frame
    │
    ▼
YOLOv8 detects faces (bounding boxes)
    │
    ▼
Crop each detected face region
    │
    ▼
face_recognition encodes the cropped face (128-d vector)
    │
    ▼
Compare encoding against your known face database
    │
    ▼
Label: "YOU" (match < 0.5 distance) or "UNKNOWN"
```

---


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## 📦 Cell 1 — Environment Check & Library Imports

This cell does two things:
1. **Lists all input files** available in the Kaggle environment (from `/kaggle/input/`)
2. Sets up the working directory for output files

The `os.walk()` traversal is a standard Kaggle boilerplate — useful to confirm that any attached datasets are visible before processing begins.


In [ ]:
import face_recognition
import os
from PIL import Image
import cv2
import numpy as np
from ultralytics import YOLO

## 📦 Cell 2 — Import Core Libraries

| Library | Purpose |
|---|---|
| `face_recognition` | Wraps **dlib**'s deep-learning face encoder; produces 128-dimensional face vectors |
| `os` | File system navigation (listing folders, joining paths) |
| `PIL.Image` | Opening, converting, and saving images |
| `cv2` (OpenCV) | Real-time webcam capture, image drawing, color-space conversion |
| `numpy` | Array math — used to compute minimum face distances |
| `ultralytics.YOLO` | Loads and runs YOLOv8 models for bounding-box detection |

> **Note:** `face_recognition` requires `dlib` and `cmake` to be installed. On Kaggle these are pre-installed; on a local machine run `pip install face_recognition` (may need `cmake` first).


In [ ]:
input_folder = "my_faces"
output_folder = "ROI_face"

os.makedirs(output_folder, exist_ok=True)

for img_name in os.listdir(input_folder):
    img_path = os.path.join(input_folder, img_name)

    image = face_recognition.load_image_file(img_path)

    face_locations = face_recognition.face_locations(image)

    for i, (top, right, bottom, left) in enumerate(face_locations):
        face_image = image[top:bottom, left:right]

        pil_image = Image.fromarray(face_image)

        save_path = os.path.join(
            output_folder,
            f"{os.path.splitext(img_name)[0]}_face_{i}.jpg"
        )

        pil_image.save(save_path)

print("Done cropping faces!")


## ✂️ Cell 3 — Crop Faces from Your Reference Images

### What this cell does
It scans the `my_faces/` folder (your personal reference photos) and crops out each detected face, saving the results to `ROI_face/` (Region of Interest).

### Why pre-crop?
The `face_recognition` library works most accurately on tightly cropped faces. Feeding it full-body images wastes compute and can produce worse encodings.

### Step-by-step breakdown
```
my_faces/photo.jpg
    │
    ├─ face_recognition.load_image_file()   → loads as RGB numpy array
    ├─ face_recognition.face_locations()    → returns (top, right, bottom, left) tuples
    │       uses HOG (fast) or CNN (accurate) under the hood
    ├─ image[top:bottom, left:right]        → numpy slicing to crop the face
    ├─ Image.fromarray()                    → converts numpy → PIL image
    └─ pil_image.save()                     → saves as JPEG to ROI_face/
```

### Output file naming
Each saved file follows the pattern: `{original_name}_face_{index}.jpg`  
e.g., `john_photo_face_0.jpg`, `john_photo_face_1.jpg` (if multiple faces found)


In [ ]:
# -------------------------
# 2. Load your face encodings
# -------------------------
known_encodings = []
dataset_path = "ROI_face"

print("[INFO] Loading your face dataset...")

for img_name in os.listdir(dataset_path):
    img_path = os.path.join(dataset_path, img_name)

    image = face_recognition.load_image_file(img_path)
    encodings = face_recognition.face_encodings(image)

    if len(encodings) > 0:
        known_encodings.append(encodings[0])

print(f"[INFO] Loaded {len(known_encodings)} face samples")


## 🔢 Cell 4 — Build the Known Faces Database (Encoding Phase)

### What is a face encoding?
`face_recognition` uses a **ResNet-based deep neural network** (trained by dlib) to convert any face image into a **128-dimensional numerical vector**. Faces of the same person cluster together in this vector space; different people produce vectors far apart.

### Why store encodings instead of images?
- **Speed**: Comparing two 128-d vectors is near-instant (vs. pixel comparison)
- **Size**: Each encoding is just 128 floats (~1KB) regardless of image resolution
- **Robustness**: Encodings are somewhat invariant to lighting and angle

### What this cell does
```
ROI_face/
    │
    ├─ load each cropped face image
    ├─ face_encodings() → 128-d numpy array
    └─ append to known_encodings list
```

> ⚠️ **If `len(encodings) == 0`** the face wasn't detected in the crop — this happens with extreme angles or very small images. Those are silently skipped. Make sure `my_faces/` contains clear, front-facing photos for best results.


In [ ]:
# -------------------------
# 1. Load YOLO model (face detector)
# -------------------------
model = YOLO("yolov8n.pt") 

## 🤖 Cell 5 — Load YOLOv8 Model

### Why use YOLO for face detection?
`face_recognition.face_locations()` is accurate but slow (~1-2 fps on CPU for a full frame). **YOLOv8n** (nano) is extremely fast (~30+ fps on CPU), making it far better suited for real-time webcam use.

### Model: `yolov8n.pt`
| Property | Value |
|---|---|
| Architecture | YOLOv8 Nano |
| Size | ~6MB |
| Speed | Fastest YOLOv8 variant |
| Trade-off | Slightly less accurate than larger variants (s/m/l/x) |

> **Note:** This model is a **general object detector** (trained on COCO — 80 classes). It can detect people/objects but is not face-specific. For higher face detection accuracy, you could replace it with a dedicated face-detection YOLO model (e.g., YOLOv8-face).

The model auto-downloads from Ultralytics on first run if not already cached.


In [ ]:
# -------------------------
# 3. Start webcam
# -------------------------
cap = cv2.VideoCapture(0)

print("[INFO] Running YOLO + Face Recognition... Press Q to quit")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame)[0]

    for box in results.boxes.xyxy:
        x1, y1, x2, y2 = map(int, box)

        # crop face from YOLO detection
        face_crop = frame[y1:y2, x1:x2]

        if face_crop.size == 0:
            continue

        # convert BGR → RGB
        rgb_face = cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB)

        encodings = face_recognition.face_encodings(rgb_face)

        label = "UNKNOWN"
        color = (0, 0, 255)

        if len(encodings) > 0 and len(known_encodings) > 0:
            face_encoding = encodings[0]

            distances = face_recognition.face_distance(known_encodings, face_encoding)
            best_match = np.min(distances)

            if best_match < 0.5:
                label = "YOU"
                color = (0, 255, 0)

        # draw box
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(frame, label, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

    cv2.imshow("YOLO + Face Recognition", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()